# Feature selection 

Here’s a practical way to (1) label feature groups in that cohort dataframe and (2) shrink to ~100 features while keeping the groups balanced and avoiding leakage.

The below code

- 1. classifies columns into feature groups: 

    - Medications (your redbook-derived med_has_*)
    - Comorbidity/ICD flags (has_*, icd*, diagnosis group flags)
    - Procedures (has_procgrp_*, proc*, CPT/HCPCS group flags)
    - Utilization / claims counts (claims, visits, etc.; baseline-year only)
    - Cost features (baseline-year costs, quarterly costs, trend/stats: cost, quarter, skew, kurt, cv, etc.)
    - Demographics (age, sex, region, plan fields)
    - Other / residual

- 2. produces:
    - a group summary table (counts + sample column names)
    - a proposed 100-feature shortlist using a fast, reproducible heuristic:
        - within each group: pick top features by univariate association (mutual information for numeric + chi-ish via MI for binaries)
        - enforce a per-group quota (so cost doesn’t dominate)
        - optionally drop highly correlated numeric features to diversify
 

## I25 / I50: Dual-year outcome labels (2018 + 2019)

For I25 and I50 we **replace** the single-year outcome labels with dual-year labels: an enrollee is positive (top 2% / 5% / 10%) if they are in that percentile in **either** 2018 or 2019. This reuses the same logic as `build_annual_costs_and_labels()` on claims, then overwrites the label columns in the existing feature parquets so we do not need to rebuild the full cohorts.

Run the cell below to:
1. Load claims for I25 and I50
2. Compute dual-year labels (2018+2019) using the same RX cost / deflation logic as the build script
3. Load existing feature parquet(s) for each code, drop old `top_*_pct_cost_2018` columns, merge in the new labels, and save (overwriting)

In [8]:
# Dual-year labels for I25 / I50: create new datasets with top_*_pct_cost_2018_2019, replacing top_*_pct_cost_2018
import sys
from pathlib import Path

# Ensure we can import build_misc_condition_features (same dir as notebook)
_nb_dir = Path.cwd()
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

from build_misc_condition_features import (
    start_spark,
    build_annual_costs_and_labels,
)

import shutil
import pandas as pd

# Paths: claims (for label computation), feature parquets (read + overwrite)
BASE_DIR = Path("/Users/charles/DATA/misc_conditions")  # claims: {BASE_DIR}/I25_claims, I50_claims
FEATURES_DIR = Path("/Users/cat2510/my_projects/misc_conditions/misc_conditions_features_with_meds")  # or misc_conditions_features_with_meds
baseline_year = 2017
outcome_year = 2018
outcome_years = [2018, 2019]
inflation = {2017: 1.0, 2018: 1.0685946832951803, 2019: 1.0685946832951803 * 1.02}

OLD_LABEL_COLS = ["top_10_pct_cost_2018", "top_5_pct_cost_2018", "top_2_pct_cost_2018"]

spark = start_spark(driver_memory="16g", num_threads=4)

for code in ["I25", "I50"]:
    claims_path = BASE_DIR / f"{code}_claims"
    if not claims_path.exists():
        print(f"[SKIP] {code}: claims not found at {claims_path}")
        continue
    print(f"\n--- {code} ---")
    df_claims = spark.read.format("parquet").load(str(claims_path))
    labels_df, thresholds = build_annual_costs_and_labels(
        df_claims, baseline_year, outcome_year, inflation,
        outcome_years=outcome_years,
    )
    labels_pd = labels_df.toPandas()
    print(f"  Dual-year labels: {len(labels_pd):,} rows")

    # New columns to add from labels (ENROLID + label cols + 2019 cost; aliases for downstream)
    new_cols = [c for c in labels_pd.columns if c != "ENROLID" and (
        c.startswith("top_") or c == "annual_cost_2019_deflated"
    )]

    # Update every feature parquet that exists for this code
    for stem in [f"{code}_features_2017_2018_with_meds", f"{code}_features_2017_2018", f"{code}_features_2017_2018_100feat"]:
        feat_path = FEATURES_DIR / f"{stem}.parquet"
        if not feat_path.exists():
            continue
        df = pd.read_parquet(feat_path)
        drop = [c for c in OLD_LABEL_COLS if c in df.columns]
        df = df.drop(columns=drop, errors="ignore")
        df = df.merge(labels_pd[["ENROLID"] + new_cols], on="ENROLID", how="left")
        df[new_cols] = df[new_cols].fillna(0)
        # If path is a directory (e.g. Spark wrote a folder), remove it so we can write a single file
        if feat_path.exists():
            if feat_path.is_dir():
                shutil.rmtree(feat_path)
            else:
                feat_path.unlink()
        df.to_parquet(feat_path, index=False)
        print(f"  Updated {feat_path.name} -> {len(df):,} rows")

spark.stop()
print("\nDone. I25/I50 parquets now have top_*_pct_cost_2018_2019 (and top_*_pct_cost_2018 alias).")


--- I25 ---


  Dual-year labels: 556,964 rows
  Updated I25_features_2017_2018_with_meds.parquet -> 425,576 rows
  Updated I25_features_2017_2018_100feat.parquet -> 425,576 rows

--- I50 ---


  Dual-year labels: 177,126 rows
  Updated I50_features_2017_2018_with_meds.parquet -> 137,357 rows
  Updated I50_features_2017_2018_100feat.parquet -> 137,357 rows

Done. I25/I50 parquets now have top_*_pct_cost_2018_2019 (and top_*_pct_cost_2018 alias).


# Quota filter for 100 features per cohort

In [1]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

codes = ["F32", "I25"]
for code in codes:
    PATH = f"misc_conditions_features_with_meds/{code}_features_2017_2018_with_meds.parquet"

    df = pd.read_parquet(PATH)

    print(f"Rows={len(df):,}  Total cols={df.shape[1]:,} ")


Rows=1,340,266  Total cols=148 
Rows=425,576  Total cols=153 


In [7]:
import sys
sys.path.insert(0, "..")  # or path to parent with public
from public.model_IAI import get_bin_flag_columns, get_cat_columns, get_true_num_columns
from curated_vs_random_1to1_multi_cohort_oct import pick_target_and_features

codes = ["E66"]
for code in codes:
    PATH = f"misc_conditions_features_with_meds/{code}_features_2017_2018_100feat.parquet"
    df = pd.read_parquet(PATH)
    print([col for col in df.columns if "2018" in col])
    print(f"Rows={len(df):,}  Total cols={df.shape[1]:,}")

    # Feature selection (match precompute / curated)
    target_col, feat_cols = pick_target_and_features(df, baseline_year=2017, outcome_year=2018)
    BIN_FLAG_COLUMNS = get_bin_flag_columns(df)
    CAT_COLUMNS = get_cat_columns(df)
    TRUE_NUM_COLUMNS = get_true_num_columns(df, CAT_COLUMNS, BIN_FLAG_COLUMNS)

    bin_in_feature = [c for c in BIN_FLAG_COLUMNS if c in feat_cols]
    non_binary = []
    for col in bin_in_feature:
        uniq = df[col].dropna().unique()
        n_uniq = len(uniq)
        if n_uniq > 2:
            non_binary.append((col, n_uniq, sorted(uniq.tolist())[:15]))  # cap list for display

    print(f"\n{code}: {len(bin_in_feature)} binary cols in features, {len(non_binary)} have >2 levels:")
    for col, n, vals in non_binary:
        print(f"  {col}: n_unique={n}  values={vals}")

['top_2_pct_cost_2018']
Rows=2,454,016  Total cols=102

E66: 65 binary cols in features, 4 have >2 levels:
  condition_procedure_total_increasing_quarters_2017: n_unique=4  values=[0, 1, 2, 3]
  direct_condition_total_increasing_quarters_2017: n_unique=4  values=[0, 1, 2, 3]
  direct_condition_quarterly_kurtosis_2017: n_unique=297235  values=[-2.000000000000001, -2.0000000000000004, -2.0, -1.9999999999999998, -1.9999999999999996, -1.9999999999999993, -1.999999999999524, -1.999999999973733, -1.9999999995735105, -1.9999999995248232, -1.9999999989472967, -1.999999998689213, -1.9999999983386596, -1.9999999982024095, -1.999999997783323]
  condition_procedure_quarterly_kurtosis_2017: n_unique=1805329  values=[-2.000000000000001, -2.0000000000000004, -2.0, -1.9999999999999998, -1.9999999999999996, -1.9999999999999993, -1.9999999992727286, -1.9999999989472967, -1.9999999973900642, -1.999999996261499, -1.9999999960287589, -1.9999999960177646, -1.9999999953743286, -1.9999999953127672, -1.9999999

In [ ]:
for col in bin_in_feature:
    uniq = df[col].dropna().unique()
    if len(uniq) > 2 and (col.startswith("has_") or "is_" in col.lower()):
        print(f"  {col}: n_unique={len(uniq)}  values={sorted(uniq.tolist())}")

In [10]:

code = "I50"
PATH = f"misc_conditions_features_with_meds/{code}_features_2017_2018_with_meds.parquet"
# ---------------------------
# Load + target + leakage guard
# ---------------------------
df = pd.read_parquet(PATH)
df.top_2_pct_cost_2018.value_counts(normalize=False),df.top_2_pct_cost_2018_2019.value_counts()

(top_2_pct_cost_2018
 0    132727
 1      4630
 Name: count, dtype: int64,
 top_2_pct_cost_2018_2019
 0    132727
 1      4630
 Name: count, dtype: int64)

In [11]:

import os
import sys
import re
import time
import math
import argparse
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

# repo imports (mirrors your other scripts)
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, parent_dir)
from public.model_IAI import (
    train_test_split_enrol,
)# split
target_col = "top_2_pct_cost_2018"
train_ids, test_ids, train_pd, test_pd = train_test_split_enrol(
    df,
    target_col=target_col,
    test_size=0.3,
    verbose=False,
    random_state=123,
)
val_ids, test_ids, val_pd, test_pd = train_test_split_enrol(
    test_pd,
    target_col=target_col,
    test_size=0.5,
    verbose=False,
    random_state=123,
)
nP = int((train_pd[target_col] == 1).sum())
nN = int((train_pd[target_col] == 0).sum())
print(f"  Split sizes: train={train_pd.shape}, val={val_pd.shape}, test={test_pd.shape}")
print(f"  Train prevalence: positives={nP:,}, controls={nN:,}, ratio={nN/max(nP,1):.2f}:1")


  Split sizes: train=(96149, 153), val=(20604, 153), test=(20604, 153)
  Train prevalence: positives=3,241, controls=92,908, ratio=28.67:1


In [33]:

code = "I50"
PATH = f"misc_conditions_features_with_meds/{code}_features_2017_2018_with_meds.parquet"
# ---------------------------
# Load + target + leakage guard
# ---------------------------
df = pd.read_parquet(PATH)

outcome_year = 2018
target_col = f"top_2_pct_cost_{outcome_year}"
annual_col = f"annual_cost_{outcome_year}_deflated"

if target_col not in df.columns:
    if annual_col in df.columns:
        thr = float(df[annual_col].quantile(0.98))
        df[target_col] = (df[annual_col] >= thr).astype(int)
        print(f"Created {target_col} from {annual_col} @98th pct = {thr:,.2f}")
    else:
        raise ValueError(f"Need either {target_col} or {annual_col}")

exclude = ["ENROLID", target_col] + [c for c in df.columns if str(outcome_year) in c]
feature_cols = [c for c in df.columns if c not in exclude]

print(f"Rows={len(df):,}  Total cols={df.shape[1]:,}  Leakage-safe features={len(feature_cols):,}")

# ---------------------------
# Feature group rules (edit these once and reuse across cohorts)
# ---------------------------
def group_feature(col: str) -> str:
    c = col.lower()

    # meds (redbook-derived)
    if col.startswith("med_has_") or c.startswith("med_"):
        return "medications"

    # procedures (your proc group one-hots)
    if col.startswith("has_procgrp_") or "procgrp" in c or c.startswith("proc_") or "cpt" in c or "hcpcs" in c:
        return "procedures"

    # ICD / diagnosis / comorbidity flags
    if col.startswith("has_") or "icd" in c or "dx" in c or "comorb" in c:
        return "diagnosis_flags"

    # cost features (baseline-year only, since we dropped outcome year)
    if ("cost" in c or "quarter" in c or "quarterly" in c or "skew" in c or "kurt" in c
        or "cv" in c or "range" in c or "increasing" in c or "decreasing" in c
        or "trend" in c or "slope" in c):
        return "cost_features"

    # utilization / counts
    if ("claims" in c or "visits" in c or "adm" in c or "ed_" in c or "er_" in c
        or "inpatient" in c or "outpatient" in c or "rx_" in c or "num_" in c or "count" in c):
        return "utilization"

    # demographics / enrollment
    if (c in {"age", "sex", "female", "male", "region"} or "zip" in c or "msa" in c
        or "plan" in c or "product" in c or "enroll" in c or "coverage" in c
        or "state" in c or "race" in c):
        return "demographics"

    return "other"


groups = {}
for c in feature_cols:
    groups.setdefault(group_feature(c), []).append(c)

# ---------------------------
# Print group summary
# ---------------------------
print("\nFeature groups (counts + sample):")
for g, cols in sorted(groups.items(), key=lambda x: -len(x[1])):
    sample = ", ".join(cols[:50])
    print(f"  {g:16s}: {len(cols):4d}  |  {sample}")

# ---------------------------
# Build a fast scoring (mutual information) to select top features
# ---------------------------
y = df[target_col].astype(int).values

def is_binary(s: pd.Series) -> bool:
    # treat bool or {0,1} as binary
    if pd.api.types.is_bool_dtype(s):
        return True
    if pd.api.types.is_numeric_dtype(s):
        u = pd.unique(s.dropna())
        if len(u) <= 3 and set(map(float, u)).issubset({0.0, 1.0}):
            return True
    return False

# Separate numeric-like vs categorical-like
num_cols = []
cat_cols = []
bin_cols = []

for c in feature_cols:
    s = df[c]
    if is_binary(s):
        bin_cols.append(c)
    elif pd.api.types.is_numeric_dtype(s):
        num_cols.append(c)
    else:
        cat_cols.append(c)

print(f"\nDetected: binary={len(bin_cols)}, numeric={len(num_cols)}, categorical={len(cat_cols)}")

# Pipeline to transform:
# - numeric: impute median
# - binary: impute 0
# - categorical: impute most_frequent + one-hot (handle_unknown)
pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
        ("bin", Pipeline([("imp", SimpleImputer(strategy="most_frequent"))]), bin_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols),
    ],
    remainder="drop",
)

X = pre.fit_transform(df[feature_cols])

# Get feature names after preprocessing
feat_names = []
feat_names.extend(num_cols)
feat_names.extend(bin_cols)
if cat_cols:
    oh = pre.named_transformers_["cat"].named_steps["oh"]
    oh_names = oh.get_feature_names_out(cat_cols).tolist()
    feat_names.extend(oh_names)

# Mutual information on transformed matrix
mi = mutual_info_classif(X, y, random_state=0, discrete_features=False)
mi_s = pd.Series(mi, index=feat_names).sort_values(ascending=False)

# Map back one-hot categorical expansions to original column group
def base_col(name: str) -> str:
    # onehot names look like "colname_value"
    for c in cat_cols:
        if name.startswith(c + "_"):
            return c
    return name

# Aggregate MI per original column (sum over onehot expansions)
mi_by_col = {}
for name, val in mi_s.items():
    b = base_col(name)
    mi_by_col[b] = mi_by_col.get(b, 0.0) + float(val)

mi_by_col = pd.Series(mi_by_col).sort_values(ascending=False)

# ---------------------------
# Choose ~100 features with group quotas + correlation pruning
# ---------------------------
TARGET_K = 100

# quota: you can tweak these; they’re a good default balance
quota = {
    "demographics": 2,
    "diagnosis_flags": 20,
    "procedures": 20,
    "medications": 20,
    "utilization": 10,
    "cost_features": 25,
    "other": 2,
}

selected = []

# pick top within each group
for g, q in quota.items():
    cols = groups.get(g, [])
    if not cols or q <= 0:
        continue
    ranked = mi_by_col.loc[mi_by_col.index.intersection(cols)]
    take = ranked.head(q).index.tolist()
    selected.extend(take)

# if we’re short, fill globally by MI
if len(selected) < TARGET_K:
    remaining = mi_by_col.index.difference(selected)
    selected.extend(remaining[: (TARGET_K - len(selected))].tolist())

# if we’re over, trim by MI
selected = list(dict.fromkeys(selected))  # dedupe preserve order
if len(selected) > TARGET_K:
    selected = mi_by_col.loc[mi_by_col.index.intersection(selected)].head(TARGET_K).index.tolist()

# Optional: prune highly correlated numeric cost-ish redundancy
# (keeps selection near 100, may drop a few)
def corr_prune(selected_cols, df, thresh=0.95):
    num_sel = [c for c in selected_cols if pd.api.types.is_numeric_dtype(df[c])]
    if len(num_sel) < 2:
        return selected_cols
    corr = df[num_sel].corr().abs()
    drop = set()
    for i in range(len(num_sel)):
        for j in range(i + 1, len(num_sel)):
            a, b = num_sel[i], num_sel[j]
            if corr.loc[a, b] >= thresh:
                # drop the lower-MI one
                mi_a = mi_by_col.get(a, 0.0)
                mi_b = mi_by_col.get(b, 0.0)
                drop.add(b if mi_a >= mi_b else a)
    return [c for c in selected_cols if c not in drop]

selected_pruned = corr_prune(selected, df, thresh=0.95)
if len(selected_pruned) > TARGET_K:
    selected_pruned = mi_by_col.loc[mi_by_col.index.intersection(selected_pruned)].head(TARGET_K).index.tolist()
elif len(selected_pruned) < TARGET_K:
    # refill if pruning dropped too many
    remaining = mi_by_col.index.difference(selected_pruned)
    selected_pruned.extend(remaining[: (TARGET_K - len(selected_pruned))].tolist())

selected = selected_pruned[:TARGET_K]

print(f"\nSelected {len(selected)} features. Top 20:")
print(pd.Series(selected[:20]).to_string(index=False))

selected_features = selected  # <- replace with your variable name (list[str])

# --- keep only columns that actually exist (safe) ---
keep_features = [c for c in selected_features if c in df.columns]
missing = sorted(set(selected_features) - set(keep_features))
if missing:
    print(f"[warn] {len(missing)} selected features not found in df (showing up to 20): {missing[:20]}")

# --- build reduced df ---
C50_100_features = df[["ENROLID", target_col] + keep_features].copy()

# optional: reorder so features after target
C50_100_features = C50_100_features[["ENROLID", target_col] + keep_features]

print(f"{code}_100_features shape:", C50_100_features.shape)
print("pos rate:", C50_100_features[target_col].mean())

# --- save if desired ---
C50_100_features.to_parquet(f"misc_conditions_features_with_meds/{code}_features_2017_2018_100feat.parquet", index=False)


Rows=137,357  Total cols=149  Leakage-safe features=144

Feature groups (counts + sample):
  diagnosis_flags :   49  |  has_cond_icd3_I50, has_comorb_icd3_E11, has_comorb_icd3_N18, has_comorb_icd3_I10, has_comorb_icd3_E78, has_comorb_icd3_I25, has_comorb_icd3_A41, has_comorb_icd3_D63, has_comorb_icd3_I48, has_comorb_icd3_D50, has_comorb_icd3_J96, has_comorb_icd3_E87, has_comorb_icd3_E66, has_comorb_icd3_M54, has_comorb_icd3_I11, has_comorb_icd3_I21, has_comorb_icd3_Z79, has_comorb_icd3_G47, has_comorb_icd3_R07, has_comorb_icd3_I42, has_comorb_icd3_R06, has_comorb_icd3_I13, has_comorb_icd3_M25, has_comorb_icd3_N17, has_comorb_icd3_N25, has_comorb_icd3_D64, 2017Q1_comorbidity_only_cost_3month, 2017Q2_comorbidity_only_cost_3month, 2017Q3_comorbidity_only_cost_3month, 2017Q4_comorbidity_only_cost_3month, comorbidity_only_cost_annual, comorbidity_only_cost_deriv_Q1_Q2_2017, comorbidity_only_cost_deriv_Q2_Q3_2017, comorbidity_only_cost_deriv_Q3_Q4_2017, comorbidity_only_is_increasing_Q1_Q2_2